# Bag-of-Words SL

Sample script to exemplify **instantiating** and **supervised-learning** on 
the bag-of-words dataset. 

In [ ]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.sl import BagOfWordsSLConfig

repo_root = get_repo_base()
device = torch.device("cuda:0")

## Configure

In [ ]:
config = BagOfWordsSLConfig.get_canonical(
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-sl-example",
    corr=0.01,
    num_words=15,
    num_samples=50_000,
    aux_words_ratio=0.5,
    prompt_length=128,
    filter_samples_above_n_tokens=384,
    word_decay_power=1.0,
    batch_size=64,
    eval_batch_size_multiple=4,
    lr_per_token=1.25e-7,
    backbone_lr_divisor=5.0,
    pad_to_multiple=8,
    model_name="HuggingFaceTB/SmolLM2-135M",
    train_epochs=2,
)

display(config.visualize())
print(f"Study folder: {config.study_folder}")
print(f"Dataset corr target: {config.data.corr:.4f}")
print(f"Backbone lr: {config.optimizer.lr:.3e}")
print(f"Head lr:     {config.optimizer.head_lr:.3e}")

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

In [7]:
state.run_training()

sft epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/195 [00:00<?, ?it/s]

epoch  0  train_corr_target=0.0066  train_corr_ground_truth=0.0458  pred_norm=1.8034  val_corr_target=0.0079  val_corr_ground_truth=0.1106


sft epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 1:   0%|          | 0/195 [00:00<?, ?it/s]

epoch  1  train_corr_target=0.0190  train_corr_ground_truth=0.0721  pred_norm=0.5568  val_corr_target=-0.0017  val_corr_ground_truth=0.1279


## Results

In [8]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

epoch,train_corr_target,train_mse_target,train_beta_target,train_corr_ground_truth,train_mse_ground_truth,train_beta_ground_truth,val_corr_target,val_mse_target,val_beta_target,val_corr_ground_truth,val_mse_ground_truth,val_beta_ground_truth
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,0.006618,1.018757,0.036417,0.045829,0.032582,0.002526,0.007888,1.02821,0.044215,0.110644,0.031508,0.006196
1,0.019023,1.018992,0.097047,0.072116,0.037796,0.003685,-0.001699,1.011116,-0.015817,0.127861,0.011357,0.01189


In [9]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()

model_preds,ground_truth,target
f64,f64,f64
0.084961,0.011353,-0.589844
0.054932,0.022217,-0.474609
0.07666,0.007019,1.15625
0.186523,0.003128,0.112793
-0.003891,-0.008972,2.453125
